In [13]:
%pip install -r requirements.txt

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached langchain-1.2.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_core-1.2.2-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_ollama-1.0.1-py3-none-any.whl.metadata (2.5 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached langgraph-1.0.5-py3-none-any.whl.metadata (7.4 kB)
  Using cached langsmith-0.5.0-py3-none-any.whl.metadata (15 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached uuid_utils-0.12.0-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (1.1 kB)
  Using cached langchain_classic-1.0.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached requ

In [102]:
import importlib
import config_prompt
importlib.reload(config_prompt)
import evaluate
importlib.reload(evaluate)

<module 'evaluate' from '/Users/kumar/Desktop/Projects.nosync/fake_reviews_prediction/Prompting/evaluate.py'>

In [103]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from config_prompt import Config
from evaluate import evaluate_model

In [104]:
Config.few_shot_prompt_template1

'\n    You are an expert at detecting truthful and deceptive reviews.\n    Task: Classify the following review of a hotel or Amazon product as either "truthful" or "deceptive".\n\n    Examples:\n    Example 1:\n    Review: "As a former Chicagoan, I\'m appalled at the Amalfi Hotel Chicago. First of all, I was expecting luxury and hospitality, neither of which I received. There\'s an Experience Designer who is supposed to be like a \'personal concierge,\' but my experience with my ED was terrible. I felt like he was trying to pressure me into staying more days than I wanted. Not only that, but I couldn\'t understand what he was saying most of the time because he was talking so fast. When I finally got to my room, I was disappointed with the quality of the furniture and the room\'s cleanliness. I had to ask for a maid to come and give me clean towels because some of the towels in the bathroom were damp. On top of that, the bed was messily done; I could have done a better job on my own bed

In [40]:
# Load your CSV
df = pd.read_csv(Config.file_path)
reviews = df["text"].tolist()

In [51]:
prompt = PromptTemplate(
    input_variables=["text"], # the placeholder to be replaced based on prompt template
    template=Config.zero_shot_prompt_template3
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [52]:
# Classify each review
results = []
for r in reviews:
    result = chain.invoke({"text": r})
    # Clean the output - extract only "truthful" or "deceptive"
    label = result.strip().lower()
    # Extract first word if model adds extra text
    if " " in label:
        label = label.split()[0]
    # Remove any punctuation
    label = label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {label}")
    results.append(label)

Raw output: Truthful. -> Cleaned: truthful
Raw output: truthful -> Cleaned: truthful
Raw output: Truthful -> Cleaned: truthful
Raw output: deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: deceptive -> Cleaned: deceptive
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: deceptive -> Cleaned: deceptive
Raw output: deceptive -> Cleaned: deceptive
Raw output: false -> Cleaned: false
Raw output: deceptive -> Cleaned: deceptive
Raw output: Truthful -> Cleaned: truthful
Raw output: truthful -> Cleaned: truthful
Raw output: deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: deceptive -> Cleaned: deceptive
Raw output: deceptive -> Cleaned: deceptive
Raw output: truthful -> Cleaned: truthful
Raw output: deceptive -> Cleaned: deceptive
Raw 

In [53]:
accuracy, f1, conf_matrix = evaluate_model(df, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

ZERO-SHOT PROMPTING RESULTS
Accuracy: 0.525

F1 Score: 0.48398898505114085

Confusion Matrix:
[[ 0  0  0  0  0  0]
 [ 0  0  0  0  0  0]
 [ 1  1 99  1  1 17]
 [ 0  0  0  0  0  0]
 [ 0  0  0  0  0  0]
 [ 0  0 91  2  0 27]]


## One-Shot Prompting
Using one example to guide the model's classification

In [96]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = Config.one_shot_prompt_template3
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [97]:
# Classify each review
one_shot_results = []
for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    # Clean the output - extract only "truthful" or "deceptive"
    label = result.strip().lower()
    # Extract first word if model adds extra text
    if " " in label:
        label = label.split()[0]
    # Remove any punctuation
    label = label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {label}")
    one_shot_results.append(label)

Raw output: Deceptive. -> Cleaned: deceptive
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: Truthful -> Cleaned: truthful
Raw output: Deceptive 

The reasons for classification are:
- Overly generic language
- Templated and sales-like wording ("superb", "friendly throughout")
- Lack of details or specificity about the hotel's service, amenities, or experiences
- Exaggeration of praise for minor aspects (special commendation)
- Marketing style -> Cleaned: deceptive
Raw output: Truthful. -> Cleaned: truthful
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Truthful -> Cleaned: truthful
Raw output: Deceptive -> Cleaned: dece

In [99]:
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS
Accuracy: 0.49583333333333335

F1 Score: 0.39739785437114816

Confusion Matrix:
[[108  12]
 [109  11]]


## Few-Shot Prompting
Using multiple examples to guide the model's classification

In [111]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template3
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [112]:
# Classify reviews using few-shot prompting
few_shot_results = []
for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    # Clean the output
    label = result.strip().lower()
    if " " in label:
        label = label.split()[0]
    label = label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {label}")
    few_shot_results.append(label)

Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Truthful. -> Cleaned: truthful
Raw output: Truthful -> Cleaned: truthful
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive -> Cleaned: deceptive
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: Truthful. -> Cleaned: truthful
Raw output: Deceptive. -> Cleaned: deceptive
Raw output: Truthful -> Cleaned: truthful
Raw output: Truthful. -> Cleaned: truthful
Raw output: Deceptive


This classification is based on several indicators:

* The reviewer mentions specific details about their experience at a hotel, such as the quality of service, room cleanliness, and price.
* However, they also mention that they plan to return and use "hilton honors" benefits later in life (30 years ago), which implies an attempt to exaggerate the value of staying with Hilton.
* Additionally, the reviewer's tone is somewhat sarcastic when mentioning their previous experiences at the hotel, us

In [113]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

FEW-SHOT PROMPTING RESULTS
Accuracy: 0.4583333333333333

F1 Score: 0.44145132455712666

Confusion Matrix:
[[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [11  2 84  1 22]
 [ 0  0  0  0  0]
 [ 1  3 89  1 26]]


## Compare All Approaches
Compare zero-shot, one-shot, and few-shot results

In [141]:
# Compare all approaches
comparison_df = pd.DataFrame({
    'Approach': ['Zero-Shot', 'One-Shot', 'Few-Shot'],
    'Accuracy': [accuracy, accuracy_one_shot, accuracy_few_shot],
    'F1 Score': [f1, f1_one_shot, f1_few_shot]
})

print("=" * 60)
print("COMPARISON OF ALL PROMPTING APPROACHES")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Find best approach
best_idx = comparison_df['Accuracy'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f"\nBest Approach: {best_approach} (Accuracy: {best_accuracy:.4f})")

COMPARISON OF ALL PROMPTING APPROACHES
 Approach  Accuracy  F1 Score
Zero-Shot      0.46  0.389417
 One-Shot      0.50  0.365804
 Few-Shot      0.50  0.479167

Best Approach: One-Shot (Accuracy: 0.5000)
